# 命令清单（2026-03 更新）

推荐主流程命令：

```powershell
pip install -r requirements.txt
python run_full_chain.py --host 127.0.0.1 --port 8000 --reload
cd web-app
npm install
npm run dev
```

模型离线链路：`session_data.py -> train_byte_session.py -> inference.py -> cluster_unknown.py`。

注意：若 `npm run dev` 或 `run_full_chain.py` 失败，请先检查依赖是否安装完整。

# 常用命令（当前代码版本）

```powershell
# 进入核心目录
cd costSensitive

# 1) 构建会话数据（可选）
python session_data.py --dataset-root dataset/iscx --output-root processed_full/sessions --num-packets 12 --packet-len 256 --train-ratio 0.8 --flow-timeout-s 10 --seed 42

# 2) 训练
python train_byte_session.py --manifest processed_full/sessions/manifest.csv --max_epoch 8 --batch-size 32
# 兼容写法（等价）
python train_byte_session.py --manifest processed_full/sessions/manifest.csv --epochs 8 --batch-size 32

# 3) 离线推理 + embedding 导出
python inference.py --manifest processed_full/sessions/manifest.csv --model-path pytorch_model/byte_session_classifier.pth --split test

# 4) 构建未知类检测器（建议）
python build_centroid_detector.py --manifest processed_full/sessions/manifest.csv --model-path pytorch_model/byte_session_classifier.pth --output-json pytorch_model/centroid_detector.json

# 5) 实时抓包推理（仅支持 live）
python main_realtime.py --mode live --source "\\Device\\NPF_{你的网卡GUID}" --model-path pytorch_model/byte_session_classifier.pth --label-map processed_full/sessions/label_map.json --output-dir pytorch_model/realtime_sessions

# 6) 全链路（API + 入库引擎 + 可选自动抓包）
cd ..
python run_full_chain.py --host 127.0.0.1 --port 8000 --reload
```